# Sky View Factor (SVF) from a DEM + Building Heights

**Inputs**

1. **Contour lines** (a polyline vector with an elevation attribute) &rarr; interpolated to a **DEM** (bare-earth terrain).
2. **Building footprints** with a **height** attribute.

**Pipeline**

```
contours --(interpolate)--> DEM --(+ building heights)--> DSM --(horizon scan)--> SVF
```

SVF is the fraction of visible sky at each ground cell (0 = fully blocked, 1 = open sky). We use the isotropic-sky approximation:

$$ \mathrm{SVF} = 1 - \frac{1}{N}\sum_{i=1}^{N} \sin^2(\beta_i) $$

where $\beta_i$ is the maximum horizon (elevation) angle within the search radius in azimuth direction $i$ (Oke 1987; Watson & Johnson 1987; Zaks&#780;ek et al. 2011).

Three back-ends are shown: **pure Python** (geopandas / rasterio / numpy / scipy), **ArcPy**, and **QGIS**. The pure-Python cells run anywhere; ArcPy/QGIS cells run only inside those environments.

## 0. Setup

Everything lives under one **`BASE`** folder. Each input and each output stage sits in its own subfolder; the output subfolders are created automatically. Edit only `BASE` and the two input file names (and the field names) to match your data.

```
BASE/
  01_contours/  contours.shp      <- input: elevation contour lines
  02_buildings/ buildings.gpkg    <- input: footprints with a height field
  03_DEM/       dem.tif           <- output: interpolated terrain
  04_DSM/       dsm.tif           <- output: terrain + building heights
  05_SVF/       svf.tif           <- output: sky view factor
```

In [ ]:
# !pip install geopandas rasterio scipy numpy shapely matplotlib
import math
import logging
from pathlib import Path
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger("svf")

# ---- BASE folder: all inputs and outputs live under here ----------------
BASE = Path(r"D:\SVF_project")

# Inputs (each in its own subfolder under BASE)
CONTOURS  = BASE / "01_contours"  / "contours.shp"    # elevation contour lines
BUILDINGS = BASE / "02_buildings" / "buildings.gpkg"  # footprints with a height field

# Outputs (each stage in its own subfolder; created automatically below)
OUT_DEM = BASE / "03_DEM" / "dem.tif"   # interpolated terrain
OUT_DSM = BASE / "04_DSM" / "dsm.tif"   # terrain + building heights
OUT_SVF = BASE / "05_SVF" / "svf.tif"   # sky view factor

# Attribute (field) names in the input vectors
ELEV_FIELD   = "ELEV"              # contour elevation attribute (m)
HEIGHT_FIELD = "height"            # building height attribute (m)
LEVELS_FIELD = "building:levels"  # fallback: levels * storey height

# Parameters
CELL_SIZE      = 2.0      # output resolution (metres / pixel)
N_DIRS         = 16       # azimuth directions for the horizon scan
MAX_RADIUS     = 200.0    # search radius (metres)
STOREY_HEIGHT  = 3.0      # m per storey when only 'levels' is available
DEFAULT_HEIGHT = 9.0      # m when neither height nor levels is present
NODATA = -9999.0

# Create the output subfolders if they don't exist yet
for _p in (OUT_DEM, OUT_DSM, OUT_SVF):
    _p.parent.mkdir(parents=True, exist_ok=True)

logger.info("BASE folder: %s", BASE)

### Helper functions

In [ ]:
def _parse_number(value):
    """Parse an OSM height/levels tag ('12', '12 m', '12.5') to float or None."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        try:
            return float(str(value).strip().split()[0].replace(',', '.'))
        except (ValueError, IndexError):
            return None


def resolve_heights(gdf, height_field, levels_field=LEVELS_FIELD,
                    storey_height=STOREY_HEIGHT, default_height=DEFAULT_HEIGHT):
    """Numeric height per feature: height, else levels*storey, else default."""
    h = np.full(len(gdf), np.nan)
    if height_field in gdf.columns:
        parsed = gdf[height_field].map(_parse_number).to_numpy(dtype='float64')
        h = np.where(~np.isnan(parsed), parsed, h)
    if levels_field in gdf.columns:
        need = np.isnan(h)
        lv = gdf[levels_field].map(_parse_number).to_numpy(dtype='float64')
        h = np.where(need & ~np.isnan(lv), lv * storey_height, h)
    n_missing = int(np.isnan(h).sum())
    if n_missing:
        logger.info('  %d/%d buildings default to %.1f m', n_missing, len(gdf), default_height)
    return np.where(np.isnan(h), default_height, h)


def _shift2d(arr, r_off, c_off, fill=np.nan):
    """Shift a 2-D array by (r_off, c_off), padding exposed edges with fill."""
    out = np.full_like(arr, fill)
    r_src = slice(max(0, -r_off), arr.shape[0] - max(0, r_off))
    c_src = slice(max(0, -c_off), arr.shape[1] - max(0, c_off))
    r_dst = slice(max(0, r_off), arr.shape[0] - max(0, -r_off))
    c_dst = slice(max(0, c_off), arr.shape[1] - max(0, -c_off))
    out[r_dst, c_dst] = arr[r_src, c_src]
    return out

## 1. Contours &rarr; DEM (pure Python)

Contour vertices are tagged with each line's elevation and interpolated to a regular grid with `scipy.interpolate.griddata`. `method='linear'` is a Delaunay/TIN interpolation; use `'cubic'` for smoother terrain.

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from scipy.interpolate import griddata
from shapely.geometry import LineString, MultiLineString

def contours_to_dem(contours, elev_field, cell_size, out_dem, method='linear'):
    gdf = gpd.read_file(contours)
    if gdf.crs is None:
        raise ValueError('contours must have a projected (metric) CRS')
    xs, ys, zs = [], [], []
    for geom, z in zip(gdf.geometry, gdf[elev_field]):
        if geom is None or geom.is_empty or z is None:
            continue
        parts = geom.geoms if isinstance(geom, MultiLineString) else [geom]
        for part in parts:
            for x, y in part.coords:
                xs.append(x); ys.append(y); zs.append(float(z))
    xs, ys, zs = map(np.asarray, (xs, ys, zs))
    minx, miny, maxx, maxy = xs.min(), ys.min(), xs.max(), ys.max()
    ncols = int(math.ceil((maxx - minx) / cell_size))
    nrows = int(math.ceil((maxy - miny) / cell_size))
    gx = minx + (np.arange(ncols) + 0.5) * cell_size
    gy = maxy - (np.arange(nrows) + 0.5) * cell_size
    mesh_x, mesh_y = np.meshgrid(gx, gy)
    logger.info('Interpolating DEM %dx%d from %d vertices', nrows, ncols, len(xs))
    dem = griddata((xs, ys), zs, (mesh_x, mesh_y), method=method)
    if np.isnan(dem).any():
        fill = griddata((xs, ys), zs, (mesh_x, mesh_y), method='nearest')
        dem = np.where(np.isnan(dem), fill, dem)
    transform = from_origin(minx, maxy, cell_size, cell_size)
    with rasterio.open(out_dem, 'w', driver='GTiff', height=nrows, width=ncols,
                       count=1, dtype='float32', crs=gdf.crs, transform=transform,
                       nodata=NODATA) as dst:
        dst.write(dem.astype('float32'), 1)
    logger.info('DEM saved -> %s', out_dem)
    return out_dem

contours_to_dem(CONTOURS, ELEV_FIELD, CELL_SIZE, OUT_DEM)

## 2. DEM + building heights &rarr; DSM

Building footprints are rasterized to the DEM grid, burning each footprint's height in metres, then added onto the terrain. On overlaps the tallest building wins.

In [ ]:
from rasterio.features import rasterize
from rasterio.enums import MergeAlg

def add_buildings_to_dem(dem, buildings, height_field, out_dsm):
    with rasterio.open(dem) as src:
        dem_arr = src.read(1).astype('float32')
        transform, crs, profile = src.transform, src.crs, src.profile
        out_shape = dem_arr.shape
    gdf = gpd.read_file(buildings)
    if gdf.crs != crs:
        gdf = gdf.to_crs(crs)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    heights = resolve_heights(gdf, height_field)
    shapes = sorted(zip(gdf.geometry, heights), key=lambda t: t[1])  # tallest last
    bld = rasterize(shapes, out_shape=out_shape, transform=transform, fill=0.0,
                    merge_alg=MergeAlg.replace, dtype='float32')
    dsm = dem_arr + bld
    profile.update(dtype='float32', count=1, nodata=NODATA)
    with rasterio.open(out_dsm, 'w', **profile) as dst:
        dst.write(dsm.astype('float32'), 1)
    logger.info('DSM saved -> %s (max building add = %.1f m)', out_dsm, float(bld.max()))
    return out_dsm

add_buildings_to_dem(OUT_DEM, BUILDINGS, HEIGHT_FIELD, OUT_DSM)

## 3. DSM &rarr; SVF (horizon scan)

For each azimuth direction we march outward one pixel step at a time, tracking the maximum horizon tangent seen so far (vectorised over the whole array). The horizon angle per direction is combined as $\mathrm{SVF} = 1 - \overline{\sin^2(\beta)}$.

In [ ]:
def svf_from_dsm(dsm, out_svf, n_dirs=N_DIRS, max_radius=MAX_RADIUS):
    with rasterio.open(dsm) as src:
        z = src.read(1).astype('float64')
        transform, profile, nodata = src.transform, src.profile, src.nodata
    px, py = abs(transform.a), abs(transform.e)
    cell = (px + py) / 2.0
    max_steps = max(int(max_radius / cell), 1)
    if nodata is not None:
        z = np.where(z == nodata, np.nan, z)
    valid = ~np.isnan(z)
    z0 = np.where(valid, z, 0.0)
    logger.info('SVF: %s, %d dirs, %.0f m (%d steps)', z.shape, n_dirs, max_radius, max_steps)
    sin2_sum = np.zeros_like(z)
    for d in range(n_dirs):
        az = 2.0 * math.pi * d / n_dirs
        dx, dy = math.cos(az), -math.sin(az)  # dy: row index grows downward
        max_tan = np.full_like(z, -np.inf)
        for step in range(1, max_steps + 1):
            r_off, c_off = int(round(dy * step)), int(round(dx * step))
            if r_off == 0 and c_off == 0:
                continue
            shifted = _shift2d(z0, r_off, c_off, fill=np.nan)
            dist = math.hypot(c_off * px, r_off * py)
            tan_ang = np.where(np.isnan(shifted), -np.inf, (shifted - z0) / dist)
            np.maximum(max_tan, tan_ang, out=max_tan)
        horizon = np.arctan(np.where(np.isfinite(max_tan), max_tan, 0.0))
        horizon = np.clip(horizon, 0.0, math.pi / 2)
        sin2_sum += np.sin(horizon) ** 2
    svf = np.where(valid, 1.0 - sin2_sum / n_dirs, NODATA)
    profile.update(dtype='float32', count=1, nodata=NODATA)
    with rasterio.open(out_svf, 'w', **profile) as dst:
        dst.write(svf.astype('float32'), 1)
    good = svf[svf != NODATA]
    logger.info('SVF saved -> %s (range %.3f-%.3f, mean %.3f)', out_svf, good.min(), good.max(), good.mean())
    return out_svf

svf_from_dsm(OUT_DSM, OUT_SVF)

## 4. Quick-look plot

In [ ]:
import matplotlib.pyplot as plt

with rasterio.open(OUT_SVF) as src:
    svf = src.read(1)
svf = np.where(svf == NODATA, np.nan, svf)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(svf, cmap='viridis', vmin=0, vmax=1)
ax.set_title('Sky View Factor'); ax.axis('off')
fig.colorbar(im, ax=ax, shrink=0.8, label='SVF (0 = blocked, 1 = open sky)')
plt.tight_layout(); plt.show()

## Alternative A - ArcPy (ArcGIS Pro)

Requires the **Spatial Analyst** and **3D Analyst** extensions. `TopoToRaster` builds the DEM directly from contour lines; the built-in **Sky View Factor** tool computes SVF from the DSM.

In [ ]:
import arcpy
from arcpy.sa import Plus, SkyViewFactor, Con, IsNull

arcpy.CheckOutExtension('Spatial'); arcpy.CheckOutExtension('3D')
arcpy.env.cellSize = CELL_SIZE

# arcpy expects string paths
contours, buildings = str(CONTOURS), str(BUILDINGS)
dem, dsm, svf = str(OUT_DEM), str(OUT_DSM), str(OUT_SVF)

# 1. Contours -> DEM
arcpy.ddd.TopoToRaster([f'{contours} {ELEV_FIELD} Contour'], dem, CELL_SIZE)
# 2. Buildings -> height raster, then DEM + buildings
arcpy.conversion.PolygonToRaster(buildings, HEIGHT_FIELD, 'in_memory/bld_h', cellsize=CELL_SIZE)
bld = Con(IsNull('in_memory/bld_h'), 0, 'in_memory/bld_h')
Plus(dem, bld).save(dsm)
# 3. DSM -> SVF
SkyViewFactor(dsm, zenith_divisions=8, azimuth_divisions=16).save(svf)
arcpy.CheckInExtension('Spatial'); arcpy.CheckInExtension('3D')

## Alternative B - QGIS (PyQGIS Processing)

Run inside the **QGIS Python console**. Uses GDAL for the DEM/rasterize/calculator steps and the **SAGA** *Sky View Factor* algorithm for the final step (enable the SAGA provider in Processing settings).

In [ ]:
import processing

# QGIS processing expects string paths
contours, buildings = str(CONTOURS), str(BUILDINGS)
dem, dsm, svf = str(OUT_DEM), str(OUT_DSM), str(OUT_SVF)
bld_ras = str(OUT_DSM.with_name(OUT_DSM.stem + '_bld.tif'))

# 1. Contours -> DEM
processing.run('gdal:gridinversedistancenearestneighbor',
               {'INPUT': contours, 'Z_FIELD': ELEV_FIELD, 'OUTPUT': dem})
# 2. Buildings -> height raster
processing.run('gdal:rasterize',
               {'INPUT': buildings, 'FIELD': HEIGHT_FIELD, 'UNITS': 1,
                'WIDTH': CELL_SIZE, 'HEIGHT': CELL_SIZE, 'INIT': 0, 'OUTPUT': bld_ras})
# 2b. DSM = DEM + buildings
processing.run('gdal:rastercalculator',
               {'INPUT_A': dem, 'BAND_A': 1, 'INPUT_B': bld_ras, 'BAND_B': 1,
                'FORMULA': 'A + B', 'OUTPUT': dsm})
# 3. DSM -> SVF (SAGA)
processing.run('sagang:skyviewfactor',
               {'DEM': dsm, 'RADIUS': MAX_RADIUS, 'METHOD': 1, 'NDIRS': 8, 'SVF': svf})